# SESSION 3 - Vector DB & Retrieval

### Task 1: FAISS Index with Song Embeddings

In [ ]:
import faiss
import numpy as np

In [ ]:
dimension = 4
np.random.seed(42)
song_embeddings = np.random.random((5, dimension)).astype("float32")

index = faiss.IndexFlatL2(dimension)
index.add(song_embeddings)

In [ ]:
print("Total song vectors in FAISS index:", index.ntotal)

### Task 2: ChromaDB Restaurant Vector Store

In [ ]:
import chromadb

chroma_client = chromadb.Client()
restaurant_col = chroma_client.create_collection("restaurants_db")

In [ ]:
restaurants = [
    "Barbeque Nation",
    "Domino's Pizza",
    "Haldiram's",
    "Subway",
    "Burger King"
]
restaurant_ids = [f"rest_{i}" for i in range(1, 6)]
np.random.seed(15)
restaurant_embeddings = np.random.random((5, 3)).tolist()

restaurant_col.add(
    ids=restaurant_ids,
    documents=restaurants,
    embeddings=restaurant_embeddings
)

In [ ]:
query_vector = np.random.random((1, 3)).tolist()
results = restaurant_col.query(query_embeddings=query_vector, n_results=1)

print("Query Vector:", [round(x, 4) for x in query_vector[0]])
print("Most Similar Restaurant:", results["documents"][0][0])
print("Distance Score:", round(results["distances"][0][0], 4))

### Task 3: Movie Search with FAISS

In [ ]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
movie_titles = [
    "Mad Max: Fury Road",
    "John Wick",
    "The Notebook",
    "Toy Story",
    "Mission: Impossible"
]

movie_descriptions = [
    "High-octane post-apocalyptic action and vehicle combat across desert wastelands.",
    "An ex-hitman comes out of retirement seeking vengeance with intense martial arts gunfights.",
    "A heartwarming romantic drama chronicling a passionate love story enduring over decades.",
    "An animated comedy adventure about secret lives of toys when humans leave the room.",
    "A secret agent embarks on high-stakes covert missions with explosive action stunts."
]

movie_embeddings = st_model.encode(movie_descriptions).astype("float32")

In [ ]:
def find_top_movies(query, titles, embeddings, model, top_k=2):
    query_vec = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_vec)

    norm_embeddings = embeddings.copy()
    faiss.normalize_L2(norm_embeddings)

    index = faiss.IndexFlatIP(norm_embeddings.shape[1])
    index.add(norm_embeddings)

    scores, indices = index.search(query_vec, top_k)
    print(f"Top {top_k} Results for query: '{query}':")
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
        print(f"{rank}. {titles[idx]} (Similarity Score: {score:.4f})")

In [ ]:
find_top_movies("action movie", movie_titles, movie_embeddings, st_model, top_k=2)

### Task 4: Instagram Caption Retrieval in ChromaDB

In [ ]:
ig_col = chroma_client.create_collection("instagram_captions")

captions = [
    "Saturday morning coffee and peaceful relaxation",
    "Sunday brunch with best friends and endless laughter",
    "Chilling by the beach catching weekend sunsets",
    "Late night study session before exams",
    "Monday morning workout grind at the gym"
]
caption_ids = [f"cap_{i}" for i in range(1, 6)]

np.random.seed(88)
caption_embeddings = np.random.random((5, 4)).tolist()

ig_col.add(
    ids=caption_ids,
    documents=captions,
    embeddings=caption_embeddings
)

In [ ]:
query_caption = np.random.random((1, 4)).tolist()
ig_results = ig_col.query(query_embeddings=query_caption, n_results=3)

print("Top 3 most similar captions for 'Weekend vibes':")
for rank, (doc, dist) in enumerate(zip(ig_results["documents"][0], ig_results["distances"][0]), 1):
    print(f"{rank}. '{doc}' (Distance: {dist:.4f})")

**One-Line Retrieval Explanation:**
Retrieval works by calculating geometric distance (squared Euclidean or cosine distance) between the query vector and all stored caption vectors in ChromaDB, returning the captions mapped to vectors with the lowest distance values.

### Task 5: Flipkart Product Search with FAISS

**ChatGPT Prompt:**
> *"Generate 5 realistic product listings for a Flipkart-style e-commerce store with product titles and a 1-sentence description covering electronics, audio, fashion, and home categories."*

**ChatGPT Output:**
1. **boAt Rockerz 450 Pro** - Wireless Bluetooth on-ear headphones with 70 hours playtime and immersive HD audio.
2. **Sony WH-1000XM5** - Premium over-ear wireless noise cancelling headphones with crystal-clear voice calling.
3. **Puma Men's Athletic Running Shoes** - Breathable mesh sports shoes designed for long-distance jogging and gym sessions.
4. **Milton Thermosteel Water Bottle** - Double-walled insulated stainless steel bottle keeping liquids hot or cold for 24 hours.
5. **Samsung 55-inch Crystal 4K Smart TV** - Ultra HD smart LED television with dynamic crystal color and voice assistant.

In [ ]:
products = [
    "boAt Rockerz 450 Pro: Wireless Bluetooth on-ear headphones with 70 hours playtime and immersive HD audio.",
    "Sony WH-1000XM5: Premium over-ear wireless noise cancelling headphones with crystal-clear voice calling.",
    "Puma Men's Athletic Running Shoes: Breathable mesh sports shoes designed for long-distance jogging and gym sessions.",
    "Milton Thermosteel Water Bottle: Double-walled insulated stainless steel bottle keeping liquids hot or cold for 24 hours.",
    "Samsung 55-inch Crystal 4K Smart TV: Ultra HD smart LED television with dynamic crystal color and voice assistant."
]

product_embeddings = st_model.encode(products).astype("float32")
faiss.normalize_L2(product_embeddings)

In [ ]:
product_index = faiss.IndexFlatIP(product_embeddings.shape[1])
product_index.add(product_embeddings)

In [ ]:
search_query = "wireless headphones"
search_vector = st_model.encode([search_query]).astype("float32")
faiss.normalize_L2(search_vector)

distances, top_indices = product_index.search(search_vector, k=2)

print(f"Most similar products for query '{search_query}':\n")
for rank, (score, idx) in enumerate(zip(distances[0], top_indices[0]), 1):
    print(f"{rank}. {products[idx]}")
    print(f"   Similarity Score: {score:.4f}\n")